In [ ]:
# VOLVE PRODUCTION DATA CLEANING AND KPI CREATION
# Notebook 01 Purpose:
# - Load Volve production Excel data
# - Clean daily production records
# - Separate producer and injector records
# - Create production KPIs
# - Export clean CSV files for later analysis


# 1. Import libraries

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

print("Libraries loaded successfully.")


# 2. Find the Volve production Excel file
# This code checks different possible file names/locations.

possible_paths = [
    Path("../data/Volve_Production_Data.xlsx"),
    Path("../data/Volve_production_data.xlsx"),
    Path("data/Volve_Production_Data.xlsx"),
    Path("data/Volve_production_data.xlsx"),
    Path("Volve_Production_Data.xlsx"),
    Path("Volve_production_data.xlsx")
]

excel_path = None

for path in possible_paths:
    if path.exists():
        excel_path = path
        break

if excel_path is None:
    raise FileNotFoundError(
        "Volve production Excel file not found. "
        "Put the file inside the data folder and name it Volve_Production_Data.xlsx"
    )

print("Using Excel file:", excel_path)



# 3. Check workbook sheet names

excel_file = pd.ExcelFile(excel_path)

print("Workbook sheets:")
print(excel_file.sheet_names)


# 4. Load daily and monthly production sheets

daily = pd.read_excel(excel_path, sheet_name="Daily Production Data")
monthly = pd.read_excel(excel_path, sheet_name="Monthly Production Data")

print("Daily production shape:", daily.shape)
print("Monthly production shape:", monthly.shape)

display(daily.head())
display(monthly.head())


# 5. Check column names

print("Daily production columns:")
for col in daily.columns:
    print("-", col)


# 6. Missing value check on raw daily data
# This only checks missing values. It does not delete or replace anything.

missing_summary = daily.isna().sum().sort_values(ascending=False)

print("Missing values in raw daily data:")
display(missing_summary)



# 7. Select useful columns for analysis
# - production volumes, - injection volumes, - well identity, - well role, - basic pressure/choke context

columns_to_keep = [
    "DATEPRD",
    "WELL_BORE_CODE",
    "NPD_WELL_BORE_NAME",
    "ON_STREAM_HRS",
    "AVG_DOWNHOLE_PRESSURE",
    "AVG_DOWNHOLE_TEMPERATURE",
    "AVG_CHOKE_SIZE_P",
    "AVG_WHP_P",
    "AVG_WHT_P",
    "BORE_OIL_VOL",
    "BORE_GAS_VOL",
    "BORE_WAT_VOL",
    "BORE_WI_VOL",
    "FLOW_KIND",
    "WELL_TYPE"
]

# Keep only columns that exist in the workbook.
available_columns = [col for col in columns_to_keep if col in daily.columns]
missing_expected_columns = [col for col in columns_to_keep if col not in daily.columns]

if missing_expected_columns:
    print("These expected columns were not found and will be skipped:")
    print(missing_expected_columns)

daily_clean = daily[available_columns].copy()

print("Selected daily_clean shape:", daily_clean.shape)
display(daily_clean.head())


# 8. Format the production data and Convert production date to datetime

daily_clean["DATEPRD"] = pd.to_datetime(daily_clean["DATEPRD"], errors="coerce")

print("Date range:")
print("Start:", daily_clean["DATEPRD"].min())
print("End:", daily_clean["DATEPRD"].max())

print("Number of wellbores:", daily_clean["WELL_BORE_CODE"].nunique())


# 9. Remove rows with no date or no well name
# These rows cannot be used for time-based well analysis.

daily_clean = daily_clean.dropna(subset=["DATEPRD", "WELL_BORE_CODE"]).copy()

print("Shape after removing rows with missing date or well name:", daily_clean.shape)


# 10. Create row-level well role as # OP = Producer or # WI = Injector

# Note that some wells may have multiple well types over time in the oil industry. 
# producer at one time and injector at another time. so the role is assigned row by row.

daily_clean["WELL_ROLE"] = daily_clean["WELL_TYPE"].map({
    "OP": "Producer",
    "WI": "Injector"
})

print("Well role counts:")
display(daily_clean["WELL_ROLE"].value_counts(dropna=False))


# 11. Check wells that appear as both producer and injector

role_changes = daily_clean.groupby("WELL_BORE_CODE")["WELL_ROLE"].nunique()
dual_role_wells = role_changes[role_changes > 1].index.tolist()

print("Wells with more than one role:")
print(dual_role_wells)

for well in dual_role_wells:
    sub = daily_clean[daily_clean["WELL_BORE_CODE"] == well]
    
    for role in sub["WELL_ROLE"].dropna().unique():
        role_data = sub[sub["WELL_ROLE"] == role]
        print(
            well,
            "|",
            role,
            ":",
            role_data["DATEPRD"].min().date(),
            "to",
            role_data["DATEPRD"].max().date(),
            "| rows:",
            len(role_data)
        )


# 12. Convert numeric columns

numeric_columns = [
    "ON_STREAM_HRS",
    "BORE_OIL_VOL",
    "BORE_GAS_VOL",
    "BORE_WAT_VOL",
    "BORE_WI_VOL",
    "AVG_DOWNHOLE_PRESSURE",
    "AVG_DOWNHOLE_TEMPERATURE",
    "AVG_CHOKE_SIZE_P",
    "AVG_WHP_P",
    "AVG_WHT_P"
]

for col in numeric_columns:
    if col in daily_clean.columns:
        daily_clean[col] = pd.to_numeric(daily_clean[col], errors="coerce")


# 13. Keep original on-stream hours before filling blanks

daily_clean["ON_STREAM_HRS_RAW"] = daily_clean["ON_STREAM_HRS"]


# 14. Create data quality flags before filling missing volumes. # to preserve possible real data gaps that exist before blanks are filled.

daily_clean["PRODUCER_VOLUME_GAP_FLAG"] = (
    (daily_clean["WELL_ROLE"] == "Producer") &
    (daily_clean["ON_STREAM_HRS"].fillna(0) > 0) &
    (
        daily_clean["BORE_OIL_VOL"].isna() |
        daily_clean["BORE_GAS_VOL"].isna() |
        daily_clean["BORE_WAT_VOL"].isna()
    )
).astype(int)

daily_clean["INJECTOR_VOLUME_GAP_FLAG"] = (
    (daily_clean["WELL_ROLE"] == "Injector") &
    (daily_clean["ON_STREAM_HRS"].fillna(0) > 0) &
    (daily_clean["BORE_WI_VOL"].isna())
).astype(int)

print("Producer volume gaps flagged:", daily_clean["PRODUCER_VOLUME_GAP_FLAG"].sum())
print("Injector volume gaps flagged:", daily_clean["INJECTOR_VOLUME_GAP_FLAG"].sum())


# 15. Fill structural blanks with zero
# Technical Reasons:
# - Producers do not normally inject water, so missing WI for producer rows becomes 0.
# - Injectors do not normally produce oil/gas/water, so missing production volumes become 0.
# - Remaining missing production-volume values are filled with 0 for aggregation.
# - Pressure/choke values are NOT filled with 0.

production_cols = ["BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL"]

# Producers: missing injection volume becomes 0
daily_clean.loc[daily_clean["WELL_ROLE"] == "Producer", "BORE_WI_VOL"] = (
    daily_clean.loc[daily_clean["WELL_ROLE"] == "Producer", "BORE_WI_VOL"].fillna(0)
)

# Injectors: missing oil/gas/water volumes become 0
for col in production_cols:
    daily_clean.loc[daily_clean["WELL_ROLE"] == "Injector", col] = (
        daily_clean.loc[daily_clean["WELL_ROLE"] == "Injector", col].fillna(0)
    )

# Fill remaining production-volume blanks with 0 for aggregation
daily_clean[production_cols + ["BORE_WI_VOL"]] = daily_clean[production_cols + ["BORE_WI_VOL"]].fillna(0)

# Missing on-stream hours become 0 for safe rate calculations
daily_clean["ON_STREAM_HRS"] = daily_clean["ON_STREAM_HRS"].fillna(0)

print("Remaining missing values in main volume columns:")
display(daily_clean[production_cols + ["BORE_WI_VOL", "ON_STREAM_HRS"]].isna().sum())


# 16. Add Year, Month, and YearMonth columns

daily_clean["Year"] = daily_clean["DATEPRD"].dt.year
daily_clean["Month"] = daily_clean["DATEPRD"].dt.month
daily_clean["YearMonth"] = daily_clean["DATEPRD"].dt.to_period("M").dt.to_timestamp()

display(daily_clean[["DATEPRD", "Year", "Month", "YearMonth"]].head())


# 17. Create technical production KPIs
# KPIs: LIQUID_VOL, WATER_CUT, OIL_RATE_PER_DAY, WATER_RATE_PER_DAY, GAS_OIL_RATIO, WATER_INJECTION_RATE_PER_DAY

# Calculate rates only when ON_STREAM_HRS > 0.
# Producer KPIs for producer rows. # Injector KPIs for injector rows.

producer_mask = daily_clean["WELL_ROLE"] == "Producer"
injector_mask = daily_clean["WELL_ROLE"] == "Injector"

# Liquid volume for producer rows
daily_clean["LIQUID_VOL"] = np.nan
daily_clean.loc[producer_mask, "LIQUID_VOL"] = (
    daily_clean.loc[producer_mask, "BORE_OIL_VOL"] +
    daily_clean.loc[producer_mask, "BORE_WAT_VOL"]
)

# Water cut for producer rows only
daily_clean["WATER_CUT"] = np.where(
    producer_mask & (daily_clean["LIQUID_VOL"] > 0),
    daily_clean["BORE_WAT_VOL"] / daily_clean["LIQUID_VOL"],
    np.nan
)

# Oil rate for producer rows only
daily_clean["OIL_RATE_PER_DAY"] = np.where(
    producer_mask & (daily_clean["ON_STREAM_HRS"] > 0),
    daily_clean["BORE_OIL_VOL"] / (daily_clean["ON_STREAM_HRS"] / 24),
    np.nan
)

# Water rate for producer rows only
daily_clean["WATER_RATE_PER_DAY"] = np.where(
    producer_mask & (daily_clean["ON_STREAM_HRS"] > 0),
    daily_clean["BORE_WAT_VOL"] / (daily_clean["ON_STREAM_HRS"] / 24),
    np.nan
)

# Gas-oil ratio (GOR) for producer rows only
daily_clean["GAS_OIL_RATIO"] = np.where(
    producer_mask & (daily_clean["BORE_OIL_VOL"] > 0),
    daily_clean["BORE_GAS_VOL"] / daily_clean["BORE_OIL_VOL"],
    np.nan
)

# Injection rate is meaningful for injector rows only
daily_clean["WATER_INJECTION_RATE_PER_DAY"] = np.where(
    injector_mask & (daily_clean["ON_STREAM_HRS"] > 0),
    daily_clean["BORE_WI_VOL"] / (daily_clean["ON_STREAM_HRS"] / 24),
    np.nan
)

display(daily_clean.head())


# 18. Create monthly field summary. # This table summarises total field production and injection by month.

monthly_field_summary = (
    daily_clean
    .groupby("YearMonth", as_index=False)
    .agg({
        "BORE_OIL_VOL": "sum",
        "BORE_GAS_VOL": "sum",
        "BORE_WAT_VOL": "sum",
        "BORE_WI_VOL": "sum",
        "ON_STREAM_HRS": "sum"
    })
)

monthly_field_summary["LIQUID_VOL"] = (
    monthly_field_summary["BORE_OIL_VOL"] +
    monthly_field_summary["BORE_WAT_VOL"]
)

monthly_field_summary["WATER_CUT"] = np.where(
    monthly_field_summary["LIQUID_VOL"] > 0,
    monthly_field_summary["BORE_WAT_VOL"] / monthly_field_summary["LIQUID_VOL"],
    np.nan
)

monthly_field_summary["CUMULATIVE_OIL_VOL"] = monthly_field_summary["BORE_OIL_VOL"].cumsum()
monthly_field_summary["CUMULATIVE_GAS_VOL"] = monthly_field_summary["BORE_GAS_VOL"].cumsum()
monthly_field_summary["CUMULATIVE_WATER_VOL"] = monthly_field_summary["BORE_WAT_VOL"].cumsum()
monthly_field_summary["CUMULATIVE_WI_VOL"] = monthly_field_summary["BORE_WI_VOL"].cumsum()

display(monthly_field_summary.head())


# 19. Create monthly well KPI table. keeps producer and injector records separate using WELL_ROLE.

monthly_well = (
    daily_clean
    .groupby(["YearMonth", "WELL_BORE_CODE", "WELL_ROLE", "WELL_TYPE"], as_index=False)
    .agg({
        "BORE_OIL_VOL": "sum",
        "BORE_GAS_VOL": "sum",
        "BORE_WAT_VOL": "sum",
        "BORE_WI_VOL": "sum",
        "ON_STREAM_HRS": "sum",
        "LIQUID_VOL": "sum",
        "WATER_CUT": "mean",
        "OIL_RATE_PER_DAY": "mean",
        "WATER_RATE_PER_DAY": "mean",
        "GAS_OIL_RATIO": "mean",
        "WATER_INJECTION_RATE_PER_DAY": "mean",
        "PRODUCER_VOLUME_GAP_FLAG": "sum",
        "INJECTOR_VOLUME_GAP_FLAG": "sum"
    })
)

display(monthly_well.head())

# Flag negative water production values for quality control. # Set water cut to missing where water volume is negative

daily_clean["NEGATIVE_WATER_FLAG"] = daily_clean["BORE_WAT_VOL"] < 0
print("Negative water rows:", daily_clean["NEGATIVE_WATER_FLAG"].sum())

# The original BORE_WAT_VOL value is preserved.
daily_clean.loc[daily_clean["NEGATIVE_WATER_FLAG"], "WATER_CUT"] = np.nan

# 20. Create separate producer and injector datasets

producer_daily = daily_clean[daily_clean["WELL_ROLE"] == "Producer"].copy()
injector_daily = daily_clean[daily_clean["WELL_ROLE"] == "Injector"].copy()

producer_monthly = monthly_well[monthly_well["WELL_ROLE"] == "Producer"].copy()
injector_monthly = monthly_well[monthly_well["WELL_ROLE"] == "Injector"].copy()

print("Producer daily rows:", producer_daily.shape)
print("Injector daily rows:", injector_daily.shape)
print("Producer monthly rows:", producer_monthly.shape)
print("Injector monthly rows:", injector_monthly.shape)


# 21. Quick check: top oil-producing wells

top_oil_wells = (
    producer_daily
    .groupby("WELL_BORE_CODE", as_index=False)["BORE_OIL_VOL"]
    .sum()
    .sort_values("BORE_OIL_VOL", ascending=False)
)

print("Top oil-producing wells:")
display(top_oil_wells.head(10))


# 22. Quick check: top water injectors

top_injection_wells = (
    injector_daily
    .groupby("WELL_BORE_CODE", as_index=False)["BORE_WI_VOL"]
    .sum()
    .sort_values("BORE_WI_VOL", ascending=False)
)

print("Top water injection wells:")
display(top_injection_wells.head(10))


# 23. Export cleaned files as CSV

output_folder = Path("../data")

if not output_folder.exists():
    output_folder = Path("data")

output_folder.mkdir(exist_ok=True)

daily_clean.to_csv(output_folder / "cleaned_daily_production.csv", index=False)
monthly_field_summary.to_csv(output_folder / "monthly_field_summary.csv", index=False)
monthly_well.to_csv(output_folder / "monthly_well_kpis.csv", index=False)

producer_daily.to_csv(output_folder / "producer_daily_clean.csv", index=False)
injector_daily.to_csv(output_folder / "injector_daily_clean.csv", index=False)
producer_monthly.to_csv(output_folder / "producer_monthly_kpis.csv", index=False)
injector_monthly.to_csv(output_folder / "injector_monthly_kpis.csv", index=False)

print("Export complete:")
print("-", output_folder / "cleaned_daily_production.csv")
print("-", output_folder / "monthly_field_summary.csv")
print("-", output_folder / "monthly_well_kpis.csv")
print("-", output_folder / "producer_daily_clean.csv")
print("-", output_folder / "injector_daily_clean.csv")
print("-", output_folder / "producer_monthly_kpis.csv")
print("-", output_folder / "injector_monthly_kpis.csv")



# 24. Final notebook summary

print("Notebook 01 completed successfully.")
print("Cleaned daily records:", daily_clean.shape)
print("Monthly field records:", monthly_field_summary.shape)
print("Monthly well records:", monthly_well.shape)

## Data quality and checks
# Cleaning is okay if:
.   Rows removed: 0 or very small
.   RAW date range = CLEANED date range
.   Missing DATEPRD = 0
.   Missing WELL_BORE_CODE = 0
.   Raw total oil = cleaned total oil
.   Raw total gas = cleaned total gas
.   Raw total water = cleaned total water
.   Negative values = 0
.   Bad water cut rows = 0
.   Infinite KPI values = 0
.   All exported files exist = True

In [ ]:
# 24. DATA QUALITY VALIDATION CHECKS
# Purpose: Confirm that the cleaning process did not damage the dataset.


# 1. Check row counts before and after cleaning
print("RAW daily rows:", daily.shape[0])
print("CLEANED daily rows:", daily_clean.shape[0])
print("Rows removed:", daily.shape[0] - daily_clean.shape[0])

# Remove rows with missing DATEPRD or WELL_BORE_CODE. If rows removed = 0, that is fine.


# 2. Check date range before and after cleaning
# These should match., If they do not match, investigate why dates disappeared.
raw_start = pd.to_datetime(daily["DATEPRD"], errors="coerce").min()
raw_end = pd.to_datetime(daily["DATEPRD"], errors="coerce").max()

clean_start = daily_clean["DATEPRD"].min()
clean_end = daily_clean["DATEPRD"].max()

print("\nRAW date range:", raw_start, "to", raw_end)
print("CLEANED date range:", clean_start, "to", clean_end)


# 3. Check that key ID columns have no missing values, # These should be 0 # DATEPRD and WELL_BORE_CODE must be 0.
key_columns = ["DATEPRD", "WELL_BORE_CODE", "WELL_TYPE", "WELL_ROLE"]

print("\nMissing values in key columns:")
print(daily_clean[key_columns].isna().sum())


# 4. Check production volume totals before and after cleaning. Missing volumes were treated as 0 in cleaned data.
# Therefore, total volumes should stay the same. # If difference is large, something went wrong
volume_columns = ["BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL", "BORE_WI_VOL"]

print("\nRaw vs cleaned volume totals:")

for col in volume_columns:
    raw_total = pd.to_numeric(daily[col], errors="coerce").sum()
    clean_total = daily_clean[col].sum()
    difference = clean_total - raw_total
    
    print(col)
    print("  Raw total:    ", raw_total)
    print("  Cleaned total:", clean_total)
    print("  Difference:   ", difference)


# 5. Check for negative production or injection volumes. # Negative oil/gas/water/injection volumes should usually be 0.
# If not, inspect those rows.
print("\nNegative values check:")

for col in volume_columns:
    negative_count = (daily_clean[col] < 0).sum()
    print(col, "negative values:", negative_count)


# 6. Check water cut range. Water cut should be between 0 and 1 for producer rows.
bad_water_cut = daily_clean[
    (daily_clean["WATER_CUT"].notna()) &
    ((daily_clean["WATER_CUT"] < 0) | (daily_clean["WATER_CUT"] > 1))
]

print("\nBad water cut rows:", bad_water_cut.shape[0])

# This should be 0.


# 7. Check rate calculations. Rates should not be infinite but 0.
rate_columns = [
    "OIL_RATE_PER_DAY",
    "WATER_RATE_PER_DAY",
    "GAS_OIL_RATIO",
    "WATER_INJECTION_RATE_PER_DAY"
]

print("\nInfinite values in KPI columns:")

for col in rate_columns:
    inf_count = np.isinf(daily_clean[col]).sum()
    print(col, "infinite values:", inf_count)


# 8. Check producer and injector split. Producer + Injector rows = equal total cleaned rows
# unless there are rows where WELL_ROLE is missing.
print("\nProducer/injector split:")

print("Producer daily rows:", producer_daily.shape[0])
print("Injector daily rows:", injector_daily.shape[0])
print("Producer + injector rows:", producer_daily.shape[0] + injector_daily.shape[0])
print("Total cleaned rows:", daily_clean.shape[0])


# 9. Check monthly summaries against cleaned daily totals, Differences should be 0 or extremely small. If not, investigate the discrepancies.
print("\nMonthly field summary check:")

for col in volume_columns:
    daily_total = daily_clean[col].sum()
    monthly_total = monthly_field_summary[col].sum()
    difference = monthly_total - daily_total
    
    print(col)
    print("  Daily cleaned total:  ", daily_total)
    print("  Monthly summary total:", monthly_total)
    print("  Difference:           ", difference)


# 10. Check exported CSV files exist
expected_files = [
    "cleaned_daily_production.csv",
    "monthly_field_summary.csv",
    "monthly_well_kpis.csv",
    "producer_daily_clean.csv",
    "injector_daily_clean.csv",
    "producer_monthly_kpis.csv",
    "injector_monthly_kpis.csv"
]

print("\nExported file check:")

for file in expected_files:
    file_path = output_folder / file
    print(file, "exists:", file_path.exists())



print("\nDATA QUALITY CHECK COMPLETE.")

In [ ]:
# Inspect rows with negative water production

negative_water_rows = daily_clean[daily_clean["BORE_WAT_VOL"] < 0]

negative_water_rows[
    [
        "DATEPRD",
        "WELL_BORE_CODE",
        "WELL_ROLE",
        "WELL_TYPE",
        "ON_STREAM_HRS",
        "BORE_OIL_VOL",
        "BORE_GAS_VOL",
        "BORE_WAT_VOL",
        "BORE_WI_VOL",
        "LIQUID_VOL",
        "WATER_CUT"
    ]
]


In [ ]:
# Check how large the negative water values are

negative_water_rows["BORE_WAT_VOL"].describe()